# Phase 1.8 — P3, the forecastability probe (tile 32UNU, 115 cubes)

**The question the project is actually about.** P1 established that appearance is
present in these frozen representations, P2 that the *sign* of NDVI change is
recoverable and its *magnitude* is not, P4 that weather alone explains 0.09–0.13
of the post-(proxy)-climatology anomaly at cell level. P3 asks whether a cheap
read-out over a frozen embedding plus the weather over the horizon can say where
NDVI will **be** — and whether it beats the two baselines any operational
forecaster already has: **persistence** and **climatology**.

**Nothing here is fine-tuned, and no network is present in this process.** The
notebook opens cached `.npz`, the cubes, and the manifest. CPU only.

**What to read, and in what order.**

1. Step 4 prints the row counts per horizon *before* anything is fitted, and the
   window-boundary drop at Δ = 100 d. That shrinkage is a result about the
   benchmark, not a nuisance.
2. Step 5 prints **which rows set this table's R²**. Three frames of 1580 carry
   a midsummer cube-mean NDVI below zero — cloud that both the clear-fraction
   filter and the per-pixel mask passed — and they carry most of the squared
   error at the short horizons. Every score below has to be read against that.
3. Step 8's headline is `margin_over_control`, never the raw R², and the
   band-matched `raw_rgb_only` row is on every table because on P2's sign probe
   it beat every network.
4. Step 9 reports the **extreme/dynamic** subset separately. Stable periods
   flatter persistence and hide everything interesting.

Exit test at the bottom.


## Step 1: Install, then restart

In [ ]:
import importlib.util, os, IPython
SENTINEL = "/content/.phase1_8_installed"
try:
    import google.colab            # noqa: F401
    ON_COLAB = True
except ImportError:
    # find_spec("google.colab") is NOT equivalent: it raises rather than
    # returning None when the parent `google` package is absent.
    ON_COLAB = False

if not ON_COLAB:
    print("not on Colab: skipping the install and the restart.")
    print("Run against your own environment (pip install -r requirements.txt) "
          "and continue from Step 2.")
elif os.path.exists(SENTINEL):
    print("Already installed in this runtime, skipping.")
    print(f"(delete {SENTINEL} and re-run to force a reinstall)")
else:
    # Not -q. A pip resolution failure here is the likeliest cause of every
    # later failure, and -q hides it.
    # No satlaspretrain-models and no torch: P3 reads the CACHED embeddings
    # and imports no encoder at all. Frozen by construction, because no
    # network is present in this process.
    !pip install earthnet s3fs xarray zarr netCDF4 scikit-learn scipy joblib

    # Verify before restarting, so a broken install cannot reach the encoders.
    import subprocess, sys
    probe = ("import s3fs, xarray, zarr, netCDF4, earthnet, pandas, numpy, "
             "sklearn, scipy, joblib")
    r = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout)
        print(r.stderr)
        raise RuntimeError(
            "Install did not take. Read the pip output above for the real "
            "conflict. Do not continue: Step 5 would fail to import the probe."
        )

    open(SENTINEL, "w").write("ok")
    print("\n" + "=" * 70)
    print("INSTALL VERIFIED. RESTARTING THE RUNTIME NOW. This is expected.")
    print("When it comes back, continue from Step 2. Do not re-run this cell.")
    print("=" * 70)
    IPython.get_ipython().kernel.do_shutdown(True)

## Step 2: Bootstrap

The resolver block is character-identical to every phase notebook from 1.3 on and is pinned by `tests/test_notebook_resolver.py`.

In [ ]:
import os, sys, glob, zipfile, textwrap

REQUIRED = ["data/ndvi.py", "data/loader.py", "data/paths.py",
            "data/climatology.py", "encoders/manifest.py",
            "encoders/pipeline.py", "encoders/base.py", "encoders/frames.py",
            "encoders/raw_features.py",
            "probes/cv.py", "probes/p1_appearance.py", "probes/p2_deltas.py",
            "probes/p4_ceiling.py", "probes/p3_forecast.py",
            "tests/test_cv_folds.py", "tests/test_p2_deltas.py",
            "tests/test_p3_forecast.py", "tests/conftest.py"]
ZIP_NAME = "phase1_8_repo.zip"
PHASE = "phase1_8"
INPUT_PHASE = "phase1_2"          # resolved by the shared block; NEVER read here

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"
except ImportError:
    DRIVE = None
    print("not on Colab, assuming the repo is the current directory")

def looks_like_repo(d):
    return d and all(os.path.exists(os.path.join(d, f)) for f in REQUIRED)

REPO = None
if DRIVE:
    zips = glob.glob(f"{DRIVE}/**/{ZIP_NAME}", recursive=True)
    unzipped = [os.path.dirname(os.path.dirname(h))
                for d in ("*", "*/*", "*/*/*")
                for h in glob.glob(f"{DRIVE}/{d}/probes/cv.py")]
    unzipped = [d for d in unzipped if looks_like_repo(d)]

    if zips:
        REPO = os.path.dirname(zips[0])
        marker = os.path.join(REPO, "probes", "p3_forecast.py")
        # Re-extract when the zip is newer than what is on disk. Without this a
        # freshly uploaded zip is ignored because an old checkout sits next to
        # it, and you debug last week's code.
        stale = (not os.path.exists(marker)
                 or os.path.getmtime(zips[0]) > os.path.getmtime(marker))
        if stale:
            print(f"found {zips[0]}")
            print(f"extracting into {REPO} (zip is newer)")
            with zipfile.ZipFile(zips[0]) as zf:
                zf.extractall(REPO)
            print()
            print("=" * 70)
            print("THE NOTEBOOK FILE ON DISK WAS JUST REPLACED.")
            print("Colab is still showing the cells it opened. To pick up the")
            print("new ones: File > Open notebook > Google Drive, and open")
            print("   " + os.path.join(REPO, "notebooks"))
            print("Until you do, the .py files are new and these cells are old.")
            print("=" * 70)
        else:
            print(f"using existing checkout at {REPO} (zip is not newer)")
    elif unzipped:
        REPO = unzipped[0]
        print(f"found unzipped repo, no zip present: {REPO}")
else:
    # Off Colab, walk up from the working directory: running the notebook from
    # notebooks/ is normal and must not be mistaken for a missing checkout.
    d = os.getcwd()
    while not looks_like_repo(d) and os.path.dirname(d) != d:
        d = os.path.dirname(d)
    REPO = d

if not looks_like_repo(REPO):
    raise RuntimeError(textwrap.dedent(f"""
        Could not find the Phase 1.8 code.

        Fix, 2 minutes:
          1. Run make_zip.sh locally to build {ZIP_NAME}
          2. Open https://drive.google.com
          3. Make a NEW subfolder  My Drive / NeurIPS-CCAI-2026 / phase1_8
          4. Drag {ZIP_NAME} into it (do not unzip)
          5. Re-run this cell.

        One subfolder per phase is deliberate: deleting phase1_8/ removes
        everything Phase 1.8 created and nothing an earlier phase depends on.
        data/raw stays at the project root -- it is shared, not a phase.

        Searched under: {DRIVE}
        Needed all of: {REQUIRED}
        Resolved REPO = {REPO}
    """).strip())

os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")

from data.paths import RAW_DIR, describe_phase, phase_dir

# --- READ-ONLY inputs, resolved wherever they already live -----------------
# === RESOLVER (pinned by tests/test_notebook_resolver.py) -- BEGIN ===
# Extracted and exercised by that test against a simulated Drive tree, so
# the precedence rule below cannot silently regress into "first hit wins".
def _candidates(rel, pattern="*"):
    """Every directory on Drive that could be `rel`, with its file count.

    Searched: this checkout, then Drive one, two and three levels down. Three,
    because phases are subfolders of one project folder -- the Phase 1.2
    embeddings sit at
        MyDrive / NeurIPS-CCAI-2026 / phase1_2 / data/phase1_2/embeddings
    which is two wildcards, while the shared cubes at
        MyDrive / NeurIPS-CCAI-2026 / data/raw
    are one.
    """
    seen, out = set(), []
    cands = [os.path.join(REPO, rel)]
    if DRIVE:
        for depth in ("*", "*/*", "*/*/*"):
            cands += sorted(glob.glob(f"{DRIVE}/{depth}/{rel}"))
    for c in cands:
        c = os.path.abspath(c)
        if c in seen or not os.path.isdir(c):
            continue
        seen.add(c)
        out.append((c, len(glob.glob(os.path.join(c, pattern)))))
    return out


def _resolve(rel, pattern, label, foreign_phase=False):
    """Pick ONE directory, by evidence, and show every candidate considered.

    TAKING THE FIRST HIT IS NOT A SELECTION, and it cost a real run: a stale
    copy of data/phase1_2/embeddings sat INSIDE the phase1_3 checkout, the old
    "this checkout first" rule preferred it over the true Phase 1.2 folder, and
    the run died on a pre-schema file nobody knew was there.

    So: most files wins, and for ANOTHER phase's artefacts a directory inside
    THIS phase's checkout never beats one outside it, whatever the counts. That
    is the layout contract -- a phase reads its inputs in place and never owns
    a copy -- expressed as code rather than as a docstring.
    """
    cands = [(c, n) for c, n in _candidates(rel, pattern) if n > 0]
    if not cands:
        return os.path.join(REPO, rel), []        # the caller reports the gap
    repo_abs = os.path.abspath(REPO)

    def inside_repo(c):
        return os.path.commonpath([repo_abs, c]) == repo_abs

    # The penalty applies ONLY in a per-phase checkout. In a plain development
    # clone the repo root IS where data/phase1_2 belongs, so penalising "inside
    # the repo" there would be backwards -- and a warning that fires when
    # nothing is wrong is a warning nobody reads the second time.
    def demote(c):
        return foreign_phase and IS_PHASE_CHECKOUT and inside_repo(c)

    ranked = sorted(cands, key=lambda cn: (
        0 if demote(cn[0]) else -1,                           # outside first
        -cn[1],                                               # then the fullest
        len(cn[0]),                                           # then the shortest
    ))
    chosen = ranked[0][0]
    if len(cands) > 1:
        print(f"[resolve] {label}: {len(cands)} candidate directories hold files --")
        for c, n in ranked:
            mark = "  <- USING" if c == chosen else ""
            flag = "  [inside this checkout]" if inside_repo(c) else ""
            print(f"[resolve]     {n:>4} file(s)  {c}{flag}{mark}")
    if demote(chosen):
        print(f"[resolve] WARNING: {label} resolved INSIDE this phase's checkout:")
        print(f"[resolve]   {chosen}")
        print("[resolve] Another phase's artefacts do not belong here -- one phase")
        print("[resolve] reads another's in place and never owns a copy. This is")
        print("[resolve] almost certainly stale. Delete it and re-run Step 2 so")
        print("[resolve] the real directory is found.")
    return chosen, ranked


# Is this checkout a PHASE folder (Drive), or a plain clone (local dev)? The
# name settles it and covers both Drive layouts that have existed: the nested
# "NeurIPS-CCAI-2026/phase1_3" and the older sibling "…-2026-phase1_3".
IS_PHASE_CHECKOUT = PHASE in os.path.basename(os.path.abspath(REPO))

RAW, _raw_cands = _resolve(RAW_DIR, "*.nc", "RAW")
EMB_IN, _emb_cands = _resolve(os.path.join("data", INPUT_PHASE, "embeddings"),
                              "*.npz", "EMB_IN", foreign_phase=True)
os.makedirs(RAW, exist_ok=True)

# A phase checkout should not contain another phase's artefact tree at all,
# even an empty one: it shadows the real directory on every future run.
_intruder = os.path.join(REPO, "data", INPUT_PHASE)
if IS_PHASE_CHECKOUT and os.path.isdir(_intruder):
    print()
    print(f"[resolve] NOTE: {_intruder}")
    print(f"[resolve] exists inside the {PHASE} checkout. {INPUT_PHASE} "
          "artefacts belong in the")
    print(f"[resolve] {INPUT_PHASE} subfolder. Nothing here writes to it, but it "
          "will keep shadowing")
    print("[resolve] the real one until you delete it.")
# === RESOLVER -- END ===

# --- P3's actual inputs: the 115-cube SCALED cache -------------------------
# HANDOFF_P2 section 4: do NOT add cubes to data/raw. The Phase 1.2 cache is
# keyed to exactly those 20. P3 reads data/scaled_32UNU, which Phase 1.7 built
# and which P4's ceiling and P2's scaled table were both measured on.
TILE = "32UNU"
_scaled_rel = os.path.join("data", f"scaled_{TILE}")
CUBES, _ = _resolve(os.path.join(_scaled_rel, "raw"), "*.nc", "SCALED CUBES")
SC_EMB, _ = _resolve(os.path.join(_scaled_rel, "embeddings"), "*.npz",
                     "SCALED EMBEDDINGS")
SC_MSK, _ = _resolve(os.path.join(_scaled_rel, "masks"), "*.npz",
                     "SCALED MASKS")

# --- this phase's OWN outputs ----------------------------------------------
RESULTS = phase_dir(PHASE, "results")

n_cubes = len(glob.glob(os.path.join(CUBES, "*.nc")))
n_emb = len(glob.glob(os.path.join(SC_EMB, "*.npz")))
n_msk = len(glob.glob(os.path.join(SC_MSK, "*.npz")))
print(f"\nREPO    {REPO}")
print(f"CUBES   {CUBES}   ({n_cubes} cubes)")
print(f"SC_EMB  {SC_EMB}   ({n_emb} .npz  = cubes x 5 encoders)")
print(f"SC_MSK  {SC_MSK}   ({n_msk} .npz  per-pixel masks)")
print("        ^ READ-ONLY. Common-masking is mandatory here and cannot be")
print("          approximated from clear_frac.")
print(f"RESULTS {RESULTS}   (this phase writes here only)")
describe_phase(PHASE)

from data.ndvi import ndvi
from encoders import TIER_A
from encoders.manifest import build_manifest
from probes import cv
print(f"\nimports OK. canonical NDVI at {ndvi.__module__}, "
      f"splits at {cv.__name__}, modes {cv.MODES}")
print(f"encoder roster ({len(TIER_A)}): {TIER_A}")
print("NO encoder is imported, loaded or fine-tuned here. This notebook reads")
print("cached .npz and nothing else -- frozen by construction, CPU only.")
for f in REQUIRED:
    print(f"  ok  {f}")


# --- shell helper, defined here so it can never be skipped ------------------
# Named sh(), not run(): IPython has a %run magic. If a helper called run() is
# ever undefined, automagic silently rewrites run("...") into %run("...") and
# reports a confusing error about a missing script instead of a NameError.
import shlex, subprocess

PY = shlex.quote(sys.executable)

def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd or REPO, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {cmd}")
    print(f"[exit 0] {cmd}")

print("helper ready: sh('<shell command>')")

## Step 3: Unit tests

The invariant is **0 failed**, never a particular pass count — the count grows
every phase and a hard-coded one goes stale silently.

In [ ]:
# pytest.ini already sets addopts = -q. Passing -q again makes it -qq,
# which hides the per-file progress.
sh(f"{PY} -m pytest tests")


## Step 4: Rebuild the manifest FRESH, and index the horizons

`encoders/manifest.py` has two axis columns and they are never interchangeable:
`original_axis_index` is the embedding join key and counts **acquisitions**;
`daily_axis_index` counts **days**. P3's horizons are days. `horizon_index` calls
`p2_deltas.assert_gap_axes_disagree` on the real consecutive pairs *before* it
computes a single horizon, and then ties P3's own rows to the ratio that check
measured — not to a constant typed into the code.

The tolerance is measured rather than assumed: on a 5-day orbit lattice ±3 days
accepts exact matches only, and ±5 would move a Δ = 5 d row by 100% of its
horizon.

In [ ]:
import time
import numpy as np, pandas as pd

from data.loader import load_cube
from encoders.manifest import assert_strata_present, assert_weather_join
from probes import p2_deltas as p2
from probes import p3_forecast as p3
from probes import p4_ceiling as p4

CUBE_PATHS = sorted(glob.glob(os.path.join(CUBES, "*.nc")))
print(f"{len(CUBE_PATHS)} cubes at {CUBES}\n")

t0 = time.time()
SAMPLES = [load_cube(p, verbose=False) for p in CUBE_PATHS]
MANIFEST = build_manifest(SAMPLES, verbose=False)          # REBUILT FRESH
assert_strata_present(MANIFEST)
JOIN = assert_weather_join(MANIFEST, CUBES, verbose=False)
assert max(JOIN["max_abs_diff"].values()) == 0.0, JOIN
print(f"MANIFEST {MANIFEST.shape} | {MANIFEST.cube_id.nunique()} cubes | "
      f"tiles {sorted(MANIFEST.tile.unique())} | "
      f"years {sorted(MANIFEST.year.unique())} | {time.time() - t0:.0f}s")
print("weather join re-derived from the cubes: 0 rows off their own day\n")

# probes/cv.py on this subset: three modes run, three correctly RAISE.
for mode in ("cube", "loco", "spatial_block"):
    n = len(p4.outer_folds(MANIFEST, mode, k=5))
    print(f"  {mode:<14} {n:>3} folds")
for mode in ("year", "tile", "crossed"):
    try:
        list(cv.folds(MANIFEST, mode, k=5, verbose=False))
        raise AssertionError(f"{mode} did not raise on a single tile/year subset")
    except AssertionError as e:
        if "did not raise" in str(e):
            raise
        print(f"  {mode:<14} correctly RAISES ({type(e).__name__})")
    except Exception as e:
        print(f"  {mode:<14} correctly RAISES ({type(e).__name__})")


## Step 5: Assemble everything the run fits on

Rows, common-masked targets, the horizon weather window, the observation
covariates and the severity bins — printed before a single model is fitted.

Three things here are results in their own right: the **per-horizon retention**
(§ the window boundary at Δ = 100 d), the **per-horizon pixel survival** (P2
showed survival is not monotone in gap, so it is measured at each horizon and
never modelled), and **which rows set the R²**.

In [ ]:
t0 = time.time()
p3.open_run_log(os.path.join(phase_dir(p3.PHASE, "logs"), "p3_run.log"))
DATA = p3.build_p3_data(MANIFEST, CUBES, emb_dir=SC_EMB, mask_dir=SC_MSK,
                        verbose=True)
print(f"\n[p3] inputs assembled in {(time.time() - t0) / 60:.1f} min")


## Step 6: What is being fitted, and what is deliberately not tuned

Every hyperparameter comes from `p4_ceiling.make_estimator` at its fixed
a-priori value (ridge α = D on standardised features). **There is no selection
loop, so there is nothing to prove clean about one** — strictly stronger than a
guarded loop. Standardisation is fitted on train and applied to test, per fold.

`fit_readout` takes `train_pos` as a required positional argument, so the
read-out cannot be fitted on everything by omitting one. Both poison tests are in
`tests/test_p3_forecast.py`: held-out poison must leave the fit bit-identical,
and a *training* poison must move it.

In [ ]:
import inspect

p4.describe_estimators(64)
print()
print(f"forecast rows      : {p3.FORECAST_ESTIMATORS}")
print(f"weather-only rows  : {p3.WEATHER_ONLY_ESTIMATORS}   (P4's own family)")
print(f"every control      : {p3.CONTROL_ESTIMATORS}   "
      "(so margin_over_control compares like with like)")
print()
print("the five mandatory baselines :", p3.BASELINE_KINDS)
print("the three controls           :", p3.CONTROL_KINDS)
print()
sig = inspect.signature(p3.fit_readout)
assert sig.parameters["train_pos"].default is inspect.Parameter.empty
print("fit_readout        ", sig)
print("fit_gap_control    ", inspect.signature(p2.fit_gap_control),
      "   <- P3's horizon control, imported from P2")
print("doy_climatology... ", inspect.signature(p4.doy_climatology_within_fold),
      "   <- P3's climatology baseline, imported from P4")
print()
print("context: k=3 pooled frames for every single-image encoder, ONE embedding")
print(f"for {p3.MI_ENCODER} -- its embedding at t already aggregates up to 8")
print("preceding frames, and context_block REFUSES a 3-stack for it.")


## Step 7: The run

4 horizons × 5 encoders × 3 fold modes × 5 baselines × 3 controls, plus the two
secondary aggregations. `n_jobs` parallelises over folds and changes wall-clock
only — the ridge solve is exact, HGB and the MLP carry fixed seeds with no
validation split, and the folds come from a generator with no RNG.

In [ ]:
N_JOBS = max(1, (os.cpu_count() or 2) - 1)
t0 = time.time()
DF, DATA = p3.run_p3(MANIFEST, CUBES, emb_dir=SC_EMB, mask_dir=SC_MSK,
                     k=5, n_jobs=N_JOBS, data=DATA,
                     log_path=os.path.join(phase_dir(p3.PHASE, "logs"),
                                           "p3_run.log"),
                     verbose=True)
print(f"\n[p3] run_p3: {len(DF)} rows in {(time.time() - t0) / 60:.1f} min "
      f"on {N_JOBS} workers")
DF = p3.add_margins(DF, verbose=True)


## Step 8: The table invariants

Every one of these is an assertion the exit test names, and every one of them can
fail — the tests exercise each against input built to break it.

In [ ]:
p3.assert_results_complete(DF)
p3.assert_baselines_present(DF)
p3.assert_controls_present(DF)
p3.assert_control_identical_across_views(DF)
p3.assert_climatology_rows_labelled(DF)
p3.assert_mi_flagged_and_single_frame(DF)
p3.assert_effective_n_counts_cubes(DF)
p3.assert_retention_shrinks(DATA.retention, verbose=True)
print("\nall EIGHT table invariants PASS")
print(f"  rows {DF.shape[0]} x cols {DF.shape[1]}")
print(f"  horizons      {sorted(DF.delta_days.unique())}")
print(f"  fold modes    {sorted(DF.fold_mode.unique())}")
print(f"  aggregations  {sorted(DF.aggregation.unique())}")
print(f"  model kinds   {sorted(DF.model_kind.unique())}")
print(f"  estimators    {sorted(DF.estimator.unique())}")


## Step 9: The results

`margin_over_control` is the headline, never the raw R². Read `R2pooled` under
`loco` — a held-out cube contributes two to five forecast rows there and a
per-fold R² is not a measurement, which the `n/a` in the `R2/fold` column says
out loud rather than hiding behind a mean over whichever folds happened to
survive.

In [ ]:
for mode in p3.FOLD_MODES:
    p3.print_headlines(DF, aggregation="cube_mean", fold_mode=mode)


In [ ]:
p3.print_controls(DF[DF.aggregation == "cube_mean"])


In [ ]:
for agg in ("cube_p90", "cell_mean"):
    p3.print_headlines(DF, aggregation=agg, fold_mode="cube")


### The extreme / dynamic subset

Stable periods flatter persistence: when NDVI is not moving, "predict today's
value" is nearly optimal and every model looks equal to it. The `extreme_low` and
`extreme_high` bins are where a forecast can actually differ from persistence.

In [ ]:
for mode in p3.FOLD_MODES:
    p3.print_severity_table(DF, aggregation="cube_mean", fold_mode=mode)


## Step 10: Write the table, and the cross-probe comparison

The CSV goes beside the cubes it was computed from — the placement
`scripts/scale_p4.py` and `scripts/scale_p2.py` already use — and a copy under
this phase's results directory.

In [ ]:
CSV = p3.results_path("p3_forecast_results.csv", root=os.path.dirname(CUBES))
DF.to_csv(CSV, index=False)
back = pd.read_csv(CSV)
assert back.shape == DF.shape, (back.shape, DF.shape)
print(f"wrote {CSV}  ({DF.shape[0]} rows x {DF.shape[1]} cols)")

CSV2 = p3.results_path("p3_forecast_results.csv")
DF.to_csv(CSV2, index=False)
print(f"wrote {CSV2}")
for name, table in (("retention", DATA.retention), ("survival", DATA.survival),
                    ("tolerance", DATA.tolerance), ("outliers", DATA.outliers)):
    p = os.path.join(RESULTS, f"p3_{name}.csv")
    table.to_csv(p, index=False)
    print(f"wrote {p}")
describe_phase(p3.PHASE)


In [ ]:
# Does P3's ranking match P2's refuted structural ordering? A CROSS-PROBE
# OBSERVATION only -- P2 settled that hypothesis at 115 cubes (raw_features >
# dinov2 > satlas_SI > imagenet, identical under all three fold modes,
# supported=False) and nothing here re-tests it.
P2_ORDER = ["raw_features", "dinov2_vitb14", "satlas_s2_swinb_rgb",
            "imagenet_vit_b16"]
print("P2 (delta sign, 115 cubes, stable across all three fold modes):")
print("   " + " > ".join(P2_ORDER))
print()
for mode in p3.FOLD_MODES:
    sub = DF[(DF.aggregation == "cube_mean") & (DF.fold_mode == mode)
             & (DF.estimator == "linear") & (DF.encoder != "none")
             & (DF.feature_set == "embedding")
             & (DF.encoder != p3.MI_ENCODER)]
    order = (sub.groupby("encoder").r2_pooled.mean()
             .sort_values(ascending=False).index.tolist())
    same = order == P2_ORDER
    print(f"P3 {mode:<14} " + " > ".join(order)
          + ("   <- SAME as P2" if same else "   <- DIVERGES from P2"))
print()
print(f"{p3.MI_ENCODER} is excluded from this ordering (si_comparable=False): "
      "its context is one embedding that already pools up to 8 frames.")


## Phase 1.8 is done when

- [ ] Step 3 reports **0 failed**.
- [ ] Step 4 rebuilds the manifest fresh, the weather join is 0 rows off, and
      `year` / `tile` / `crossed` still correctly RAISE.
- [ ] Step 5 prints n retained per horizon and it **shrinks** toward Δ = 100 d,
      prints per-horizon pixel survival, and prints the three frames that set
      the R².
- [ ] Step 8's **eight** table invariants pass, including `control_score`
      identical digit-for-digit across every filtered view.
- [ ] Step 9 reports the headline **and** the extreme/dynamic subset, with
      cube-clustered intervals and the effective n (CUBES) on every row.
- [ ] Every climatology row is labelled **proxy, not Stage B**. It is not H1's
      number and must never be quoted as one.
- [ ] The CSV covers 4 horizons × 5 encoders × 3 fold modes × 5 baselines ×
      3 controls, with severity bins on every row.

Then: `README.md` phase status, `log.md`, `docs/DECISIONS.md`, and
`docs/HANDOFF_CONVERGENCE.md`.
